# CreacionFeatures · vf (features acústicas propias)

El objetivo de este código es:
1. Procesar los audios de las entrevistas de acuerdo a lo descrito en el documento del trabajo
2. Calcular las features acústicas que formarán parte de los modelos posteriores

## 0. Setup
---

In [ ]:
import getpass
from pathlib import Path
from tqdm import tqdm

import pandas as pd
import numpy as np

import librosa
import soundfile as sf
import parselmouth
from parselmouth.praat import call

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Path con muestra
path_sample_base = Path(r'D:/DAIZ-WOZ/base/')

# Path con dataframe de muestra
path_df_sample = Path(getpass.getpass("Ruta con df input: "))
path_output = path_df_sample

# Speakers entrevista
name_paciente = 'Participant'
name_entrevistador = 'Ellie'

In [ ]:
##################################################
#### Parámetros de extracción de features     ####
##################################################

SR          = 16_000            # Hz — estándar ASR/speech processing
N_MFCC      = 13                # coeficientes MFCC — estándar literatura (AVEC, ComParE)
FRAME_LEN   = int(0.025 * SR)  # 25 ms — ventana de análisis estándar ETSI
HOP_LEN     = int(0.010 * SR)  # 10 ms — paso entre frames
MIN_SEG_DUR = 0.3              # s — duración mínima de segmento (evita artefactos)
F0_FMIN     = 75.0             # Hz — mínimo F0 (voces masculinas graves)
F0_FMAX     = 400.0            # Hz — máximo F0 (voces femeninas agudas)
MIN_DUR_VQ  = 0.5              # s — mínimo para análisis de calidad de voz en Praat

In [ ]:
df_sample = pd.read_csv(path_df_sample / 'df_sample.csv')
df_sample

,participant_id,phq8_binary,phq8_score,gender,split,path_audio,path_transcript,durAudio,durSession,nTurns_pat,...,ratioTurns_pat,sumPause_inter_pat,avgPause_inter_pat,stdPause_inter_pat,ratioPause_inter_pat,nPause_intra_pat,sumPause_intra_pat,avgPause_intra_pat,stdPause_intra_pat,ratioPause_intra_pat
0,300,0,2,1,test,D:/DAIZ-WOZ/base/300_P/300_AUDIO.wav,D:/DAIZ-WOZ/base/300_P/300_TRANSCRIPT.csv,648.5,584.680,58,...,0.309725,93.710,1.673393,1.786927,0.160276,29,25.330,0.873448,0.454503,0.043323
1,301,0,3,1,test,D:/DAIZ-WOZ/base/301_P/301_AUDIO.wav,D:/DAIZ-WOZ/base/301_P/301_TRANSCRIPT.csv,823.9,774.400,48,...,0.698438,37.070,0.823778,0.638970,0.047869,56,65.430,1.168393,0.526539,0.084491
2,302,0,4,1,dev,D:/DAIZ-WOZ/base/302_P/302_AUDIO.wav,D:/DAIZ-WOZ/base/302_P/302_TRANSCRIPT.csv,758.8,676.970,52,...,0.419564,97.925,1.958500,1.242940,0.144652,44,75.542,1.716864,0.914819,0.111588
3,303,0,0,0,train,D:/DAIZ-WOZ/base/303_P/303_AUDIO.wav,D:/DAIZ-WOZ/base/303_P/303_TRANSCRIPT.csv,985.3,934.100,57,...,0.715737,49.060,0.981200,1.493923,0.052521,45,46.540,1.034222,0.741526,0.049823
4,304,0,6,0,train,D:/DAIZ-WOZ/base/304_P/304_AUDIO.wav,D:/DAIZ-WOZ/base/304_P/304_TRANSCRIPT.csv,792.6,720.040,72,...,0.561969,50.870,0.726714,0.887816,0.070649,32,42.040,1.313750,1.261898,0.058386
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
181,488,0,0,0,train,D:/DAIZ-WOZ/base/488_P/488_AUDIO.wav,D:/DAIZ-WOZ/base/488_P/488_TRANSCRIPT.csv,884.9,833.100,39,...,0.650042,43.398,1.172919,1.457796,0.052092,97,121.110,1.248557,2.412238,0.145373
182,489,0,3,1,dev,D:/DAIZ-WOZ/base/489_P/489_AUDIO.wav,D:/DAIZ-WOZ/base/489_P/489_TRANSCRIPT.csv,704.7,644.098,65,...,0.327186,140.985,2.237857,1.758847,0.218887,40,54.010,1.350250,1.000673,0.083854
183,490,0,2,1,dev,D:/DAIZ-WOZ/base/490_P/490_AUDIO.wav,D:/DAIZ-WOZ/base/490_P/490_TRANSCRIPT.csv,691.3,650.478,51,...,0.392957,117.441,2.302765,1.825454,0.180546,41,74.400,1.814634,1.599202,0.114377
184,491,0,8,0,train,D:/DAIZ-WOZ/base/491_P/491_AUDIO.wav,D:/DAIZ-WOZ/base/491_P/491_TRANSCRIPT.csv,881.7,797.940,58,...,0.626839,39.167,0.870378,1.285966,0.049085,84,95.970,1.142500,1.040982,0.120272


## 1. Extracción de Features - Funciones
---

In [ ]:
## 1. Funciones auxiliares

# Media y desviación típica como únicos estadísticos funcionales
_FUNCS = ['mean', 'std']


def _functionals(x: np.ndarray, prefix: str) -> dict:
    """
    Comprime un array 1D de valores frame-a-frame en media y desviación típica.

    Se usa std poblacional (ddof=0) para consistencia cuando el número de
    frames varía entre sesiones (numpy default).
    Retorna NaN si el array está vacío o es None.
    """
    if x is None or len(x) == 0:
        return {f"{prefix}_mean": np.nan, f"{prefix}_std": np.nan}

    return {
        f"{prefix}_mean": float(np.mean(x)),
        f"{prefix}_std":  float(np.std(x)),
    }


def _empty_features() -> dict:
    """Retorna NaN para todas las features (sesiones con error)."""
    out = {}
    for prefix in ['f0', 'energy', 'centroid', 'bandwidth', 'rolloff', 'zcr',
                   'hnr', 'jitter', 'shimmer']:
        out[f"{prefix}_mean"] = np.nan
        out[f"{prefix}_std"]  = np.nan
    out['f0_voiced_frac'] = np.nan
    for i in range(1, N_MFCC + 1):
        out[f"mfcc{i:02d}_mean"] = np.nan
        out[f"mfcc{i:02d}_std"]  = np.nan
    return out


## 2. Carga de segmentos del paciente

def load_patient_segments(path_audio: str,
                           path_transcript: str,
                           speaker: str = 'Participant') -> list:
    """
    Carga el audio y extrae únicamente los segmentos de habla verbal del paciente.

    Decisión: aislar el habla del paciente via timestamps del transcript (ground
    truth anotado), en lugar de VAD automático. Esto evita contaminar las
    features con la voz de Ellie o los silencios interturno, que son
    acústicamente distintos a la habla del paciente y distorsionarían
    métricas prosódicas y de calidad vocal.

    Se excluyen marcadores no verbales (<laughter>, <cough>, <synch>, etc.)
    para no contaminar features acústicas con audio que no es habla.

    Se descartan segmentos < MIN_SEG_DUR (0.3 s) para evitar artefactos en
    el cálculo de MFCCs y estimación de F0.
    """
    df_t = pd.read_csv(path_transcript, sep='\t')
    df_t.columns = df_t.columns.str.strip()
    df_t = (df_t[df_t['speaker'] == speaker]
              .sort_values('start_time')
              .reset_index(drop=True))

    # Excluir marcadores no verbales: <laughter>, <cough>, <synch>, etc.
    df_t = df_t[~df_t['value'].str.strip().str.startswith('<', na=False)].reset_index(drop=True)

    y, _ = librosa.load(path_audio, sr=SR, mono=True)

    segments = []
    for _, row in df_t.iterrows():
        dur = row['stop_time'] - row['start_time']
        if dur < MIN_SEG_DUR:
            continue

        s = int(row['start_time'] * SR)
        e = int(row['stop_time']  * SR)
        seg = y[s:e]

        if len(seg) >= FRAME_LEN:
            segments.append(seg)

    return segments


## 3. Features espectrales (librosa)

def extract_spectral_features(segments: list) -> dict:
    """
    Features espectrales a nivel de frame sobre todos los segmentos del paciente.

    MFCCs (13 coeficientes):
        Capturan la forma del tracto vocal. Son la representación de referencia
        en reconocimiento de habla y detección de trastornos del habla.
        Los coeficientes bajos (1-4) son especialmente discriminativos para
        depresión (Williamson et al. 2016). Se usan 13 coeficientes (estándar
        MFCC basado en filterbank de 26 filtros Mel).

    Energía RMS (dB):
        Indicador de loudness. Los pacientes deprimidos muestran habla
        con menor energía y menor variabilidad energética.

    Centroide espectral:
        Centro de masa del espectro. Relacionado con el brillo tonal.
        Menor en voces más graves y monótonas (típico en depresión).

    Ancho de banda espectral:
        Dispersión del espectro en torno al centroide.

    Rolloff espectral (85%):
        Frecuencia por debajo de la cual se concentra el 85% de la energía.

    ZCR (Zero-Crossing Rate):
        Tasa de cruces por cero. Indicador de afonía y ronquera.

    Parámetros de análisis:
        - Ventana: 25 ms (estándar ETSI ES 201 108)
        - Hop: 10 ms (75% solapamiento)
        - FFT: misma longitud que ventana
    """
    all_mfcc     = []
    all_energy   = []
    all_centroid = []
    all_bw       = []
    all_rolloff  = []
    all_zcr      = []

    for seg in segments:
        if len(seg) < FRAME_LEN:
            continue

        all_mfcc.append(
            librosa.feature.mfcc(y=seg, sr=SR, n_mfcc=N_MFCC,
                                  n_fft=FRAME_LEN, hop_length=HOP_LEN)
        )

        rms = librosa.feature.rms(y=seg, frame_length=FRAME_LEN, hop_length=HOP_LEN)
        all_energy.append(20 * np.log10(rms + 1e-8))

        all_centroid.append(
            librosa.feature.spectral_centroid(y=seg, sr=SR,
                                               n_fft=FRAME_LEN, hop_length=HOP_LEN)
        )
        all_bw.append(
            librosa.feature.spectral_bandwidth(y=seg, sr=SR,
                                                n_fft=FRAME_LEN, hop_length=HOP_LEN)
        )
        all_rolloff.append(
            librosa.feature.spectral_rolloff(y=seg, sr=SR, n_fft=FRAME_LEN,
                                              hop_length=HOP_LEN, roll_percent=0.85)
        )
        all_zcr.append(
            librosa.feature.zero_crossing_rate(seg, frame_length=FRAME_LEN, hop_length=HOP_LEN)
        )

    def _cat1d(lst):
        return np.concatenate(lst, axis=1).flatten() if lst else np.array([])

    return {
        'mfcc':      np.concatenate(all_mfcc, axis=1) if all_mfcc else None,
        'energy':    _cat1d(all_energy),
        'centroid':  _cat1d(all_centroid),
        'bandwidth': _cat1d(all_bw),
        'rolloff':   _cat1d(all_rolloff),
        'zcr':       _cat1d(all_zcr),
    }


## 4. Features prosódicas y de calidad vocal (Praat via parselmouth)

def extract_praat_features(segments: list) -> dict:
    """
    F0 (pitch), HNR, jitter y shimmer via Praat para todos los segmentos.

    F0 (Fundamental Frequency):
        Frecuencia de vibración de las cuerdas vocales. Indicador prosódico
        primario en depresión: reducción de media y variabilidad (Cummins 2015).
        Se usa el estimador SHR de Praat (To Pitch) en lugar de pYIN de librosa
        por ser ~10x más rápido con precisión comparable para habla limpia.
        Solo se contabilizan frames sonoros (F0 > 0).

    HNR (Harmonics-to-Noise Ratio):
        Ratio entre componentes armónicos y ruido en la señal vocal. Mide
        la periodicidad de la voz. Valores más bajos indican mayor afonía/ronquera.
        Los pacientes deprimidos muestran HNR significativamente menor
        (Bhatt et al. 2021).

    Jitter (perturbación de periodo):
        Variación ciclo a ciclo del periodo glotal (inestabilidad de F0).
        Jitter local = |T_i - T_{i-1}| / mean(T). Mayor en voces con
        patología o tensión. Asociado positivamente con depresión.

    Shimmer (perturbación de amplitud):
        Variación ciclo a ciclo de la amplitud del pulso glotal. Indicador
        de irregularidad en la adducción de cuerdas vocales. También se
        eleva en depresión y estados de fatiga vocal.

    Mínimo para calidad vocal (MIN_DUR_VQ = 0.5 s):
        Praat necesita al menos ~3-5 períodos glotales (~30-65 ms a F0 normal)
        para estimar jitter y shimmer de forma estable. 0.5 s es conservador.
    """
    f0_voiced   = []
    n_total     = 0
    n_voiced    = 0
    hnr_list    = []
    jitter_list = []
    shimmer_list = []

    for seg in segments:
        if len(seg) < FRAME_LEN:
            continue

        snd = parselmouth.Sound(seg.astype(np.float64), sampling_frequency=SR)

        # F0 (Praat pitch tracker, SHR method)
        try:
            pitch_obj = snd.to_pitch(
                time_step=HOP_LEN / SR,
                pitch_floor=F0_FMIN,
                pitch_ceiling=F0_FMAX
            )
            f0_arr = pitch_obj.selected_array['frequency']
            n_total  += len(f0_arr)
            n_voiced += int((f0_arr > 0).sum())
            f0_voiced.extend(f0_arr[f0_arr > 0].tolist())
        except Exception:
            pass

        # HNR, Jitter, Shimmer (requieren segmento mínimo)
        if len(seg) / SR < MIN_DUR_VQ:
            continue

        try:
            harm = call(snd, "To Harmonicity (cc)", 0.01, F0_FMIN, 0.1, 1.0)
            hnr  = call(harm, "Get mean", 0, 0)
            if np.isfinite(hnr):
                hnr_list.append(hnr)
        except Exception:
            pass

        try:
            pp = call(snd, "To PointProcess (periodic, cc)", F0_FMIN, F0_FMAX)

            j = call(pp, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3)
            if np.isfinite(j):
                jitter_list.append(j)

            sh = call([snd, pp], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6)
            if np.isfinite(sh):
                shimmer_list.append(sh)
        except Exception:
            pass

    return {
        'f0':          np.array(f0_voiced),
        'voiced_frac': float(n_voiced / n_total) if n_total > 0 else np.nan,
        'hnr':         np.array(hnr_list),
        'jitter':      np.array(jitter_list),
        'shimmer':     np.array(shimmer_list),
    }


## 5. Pipeline por sesión

def extract_all_acoustic_features(row: pd.Series) -> dict:
    """
    Pipeline completo por sesión: audio + transcript → vector de features.

    Nivel de análisis: sesión (un vector por sesión).
    Decisión: con 186 sesiones, la segmentación fija (p. ej. 5 s) para DL
    inflaría artificialmente el número de muestras pero rompería la
    independencia entre observaciones en CV. El enfoque de funcionales
    sobre toda la habla del paciente es el estándar para regresión
    PHQ-8 con ML clásico (AVEC 2017 baseline, Williamson 2016, Ringeval 2017).

    Produce 45 features acústicas:
        F0 (2 + voiced_frac = 3) + Energía (2) + MFCCs 13×2 (26) +
        Espectrales 4×2 (8) + Calidad vocal 3×2 (6)
    """
    pid = row.get('participant_id', '?')
    try:
        segments = load_patient_segments(row['path_audio'], row['path_transcript'])

        if not segments:
            print(f"  [{pid}] Sin segmentos válidos")
            return _empty_features()

        sp = extract_spectral_features(segments)
        pr = extract_praat_features(segments)

        out = {}

        # F0
        out.update(_functionals(pr['f0'], 'f0'))
        out['f0_voiced_frac'] = pr['voiced_frac']

        # Energía
        out.update(_functionals(sp['energy'], 'energy'))

        # MFCCs (por coeficiente, indexados desde 1)
        if sp['mfcc'] is not None:
            for i in range(sp['mfcc'].shape[0]):
                out.update(_functionals(sp['mfcc'][i], f'mfcc{i+1:02d}'))
        else:
            for i in range(1, N_MFCC + 1):
                out[f"mfcc{i:02d}_mean"] = np.nan
                out[f"mfcc{i:02d}_std"]  = np.nan

        # Espectrales
        out.update(_functionals(sp['centroid'],  'centroid'))
        out.update(_functionals(sp['bandwidth'], 'bandwidth'))
        out.update(_functionals(sp['rolloff'],   'rolloff'))
        out.update(_functionals(sp['zcr'],       'zcr'))

        # Calidad vocal
        out.update(_functionals(pr['hnr'],     'hnr'))
        out.update(_functionals(pr['jitter'],  'jitter'))
        out.update(_functionals(pr['shimmer'], 'shimmer'))

        return out

    except Exception as e:
        print(f"  [{pid}] ERROR: {e}")
        return _empty_features()

## 2. Extracción de Features - Ejecución pipeline
---

In [ ]:
# Extracción de features acústicas — ~5-15 min dependiendo del hardware
tqdm.pandas(desc="Sesiones procesadas")

acoustic_raw = df_sample.progress_apply(extract_all_acoustic_features, axis=1)
df_features  = pd.json_normalize(acoustic_raw)

df_acoustic = pd.concat(
    [df_sample.reset_index(drop=True), df_features.reset_index(drop=True)],
    axis=1
)

print(f"Shape final:  {df_acoustic.shape}")
print(f"NaN totales:  {df_acoustic.isna().sum().sum()}")
print(f"Sesiones OK:  {df_acoustic['f0_mean'].notna().sum()} / {len(df_acoustic)}")
df_acoustic.head(3)

Sesiones procesadas:   0%|          | 0/186 [00:00<?, ?it/s]

Sesiones procesadas: 100%|██████████| 186/186 [12:35<00:00,  4.06s/it]

Shape final:  (186, 68)
NaN totales:  0
Sesiones OK:  186 / 186


,participant_id,phq8_binary,phq8_score,gender,split,path_audio,path_transcript,durAudio,durSession,nTurns_pat,...,rolloff_mean,rolloff_std,zcr_mean,zcr_std,hnr_mean,hnr_std,jitter_mean,jitter_std,shimmer_mean,shimmer_std
0,300,0,2,1,test,D:/DAIZ-WOZ/base/300_P/300_AUDIO.wav,D:/DAIZ-WOZ/base/300_P/300_TRANSCRIPT.csv,648.5,584.68,58,...,3318.896763,1569.683872,0.089543,0.058401,9.261596,4.517246,0.022831,0.012471,0.134500,0.036548
1,301,0,3,1,test,D:/DAIZ-WOZ/base/301_P/301_AUDIO.wav,D:/DAIZ-WOZ/base/301_P/301_TRANSCRIPT.csv,823.9,774.40,48,...,2584.123304,1693.424424,0.084849,0.111235,11.564567,3.137851,0.023774,0.008243,0.100201,0.025551
2,302,0,4,1,dev,D:/DAIZ-WOZ/base/302_P/302_AUDIO.wav,D:/DAIZ-WOZ/base/302_P/302_TRANSCRIPT.csv,758.8,676.97,52,...,2615.732098,1260.622413,0.062828,0.054419,8.777707,1.922941,0.022913,0.008217,0.140910,0.024773


In [ ]:
# Guardado
path_output = path_output + '/df_acoustic_features_vf.csv'
df_acoustic.to_csv(path_output, index=False)
print(f"Guardado: {path_output}  —  shape {df_acoustic.shape}")